# Model demo

Runs the large-training DSA model (`dsa_LT_seed1`, 9,736 species) on the first 7 seconds of a Xeno-Canto recording and prints the most likely species.


## Install requirements

In [ ]:
# !pip install -r ../requirements.txt

In [2]:
import torch
import sys
sys.path.append('../')

from sa4birds.utils.event_decoder import EventDecoder
from sa4birds.evaluation import load_model
from sa4birds.utils.transform import ValTransform
from registry import DT, MT, LT
import librosa
import soundfile as sf
import pandas as pd

SAMPLE_RATE = 32000
FILEPATH = "./XC973025.mp3"
DEVICE = "cpu"  # or "cuda" if a GPU is available

/path/to/sa4birds/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!wget -O XC973025.mp3 https://xeno-canto.org/973025/download

--2026-09-28 16:56:08--  https://xeno-canto.org/973025/download
Resolving xeno-canto.org (xeno-canto.org)... 145.136.250.151
Connecting to xeno-canto.org (xeno-canto.org)|145.136.250.151|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [audio/mpeg]
Saving to: ‘XC973025.mp3’

XC973025.mp3            [ <=>                ]   1.25M  --.-KB/s    in 0.1s    

2026-09-28 16:56:08 (9.68 MB/s) - ‘XC973025.mp3’ saved [1311882]



In [4]:
# eBird taxonomy of the 9,736 LT species (shipped in assets/): code -> scientific name
taxonomy = pd.read_csv('../assets/LT-species_ebird_taxonomy_v2022.csv')
code_to_scientific = taxonomy.set_index("ebird_code_v2022")["scientific_name"].to_dict()

In [5]:

ckpt = LT.get('ALL', [])[0]

# Load the trained model and its configuration from the checkpoint
model, cfg = load_model('../' + ckpt, device=DEVICE)

# attention temperature (2.0 = the value the model was trained with)
model.attention.temperature = 2.0

# Create the validation transform pipeline
# This handles spectrogram generation and preprocessing
transform = ValTransform(
        config=cfg,
        train=False,
        event_decoder=EventDecoder(extracted_interval=cfg.event_decoder.val.extracted_interval),
)

# Reverse the label map so we can convert predicted class index -> label name
label_map = { v:k for k,v in cfg.train.label_map.items()}

# Read metadata about the audio file (samplerate, duration, etc.)
file_info = sf.info(FILEPATH)
sr = file_info.samplerate

# Load the first 7 seconds of the audio file
audio, sr = sf.read(FILEPATH, start=0, stop=7*sr)


# If audio is multi-channel (e.g., stereo), convert it to mono
if audio.ndim != 1:
    audio = audio.swapaxes(1, 0)
    audio = librosa.to_mono(audio)

# Resample audio if it does not match the model's required sample rate
if sr != SAMPLE_RATE:
    audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    sr = SAMPLE_RATE

# Convert the waveform to a tensor and compute spectrogram features
# Shape transformation: (samples) -> (batch=1, channel=1, samples)
fbank_features = transform.get_spectrogram(torch.tensor(audio).unsqueeze(0).unsqueeze(0).float())

# Pad the spectrogram so it matches the expected model input size
fbank_features = transform.pad(fbank_features)

# Normalize the features using dataset mean and std from the config
fbank_features = (fbank_features - cfg.frontend.mean) / ( cfg.frontend.std * 2)

# Run inference without computing gradients (faster and uses less memory)
with torch.no_grad():
    preds = model(fbank_features.to(DEVICE))

    # Top-2 predicted classes
    top2 = preds.topk(2)
    print(f"{code_to_scientific[label_map[int(top2.indices[0][0])]]}: {top2.values[0][0]}")
    print(f"{code_to_scientific[label_map[int(top2.indices[0][1])]]}: {top2.values[0][1]}")


Milvus milvus: 0.9242955446243286
Milvus migrans: 0.7460057735443115
